# Análisis de Ventas Retail — Online Retail Dataset 2023
**Autor:** Luis Fernando Rubio Avalos  
**Fecha:** Noviembre 2026  
**Fuente:** UCI Machine Learning Repository — Online Retail Dataset

## 1. Introducción

Este proyecto analiza las ventas de una tienda de retail con operaciones en Reino Unido y mercados internacionales durante 2023.

**Preguntas de negocio:**
1. ¿Cuál es la evolución de ventas a lo largo del año?
2. ¿Qué productos generan más ingresos?
3. ¿Qué países son los mercados internacionales más valiosos?
4. ¿Qué segmentos de clientes existen y cuánto aportan?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Estilo
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--'
})
NAVY  = '#1B2A4A'
TEAL  = '#00897B'
AMBER = '#F4A100'
GRAY  = '#6C757D'
print('Librerías cargadas correctamente.')

## 2. Carga y Exploración de Datos

In [ ]:
df = pd.read_csv('online_retail_2023.csv', parse_dates=['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print('=== RESUMEN DEL DATASET ===')
print(f'Registros totales:   {len(df):,}')
print(f'Facturas únicas:     {df["InvoiceNo"].nunique():,}')
print(f'Clientes únicos:     {df["CustomerID"].nunique():,}')
print(f'Productos únicos:    {df["StockCode"].nunique():,}')
print(f'Países:              {df["Country"].nunique()}')
print(f'Período:             {df["InvoiceDate"].min().date()} → {df["InvoiceDate"].max().date()}')
print(f'Venta total:         £{df["TotalPrice"].sum():,.0f}')
print(f'Ticket promedio:     £{df.groupby("InvoiceNo")["TotalPrice"].sum().mean():.2f}')
df.head()

In [ ]:
print('Tipos de datos:')
print(df.dtypes)
print(f'\nValores nulos:\n{df.isnull().sum()}')

## 3. Análisis de Ventas Mensuales

In [ ]:
df['Month'] = df['InvoiceDate'].dt.month
monthly = df.groupby('Month').agg(
    Revenue=('TotalPrice', 'sum'),
    Orders=('InvoiceNo', 'nunique')
).reset_index()
month_labels = ['Ene','Feb','Mar','Abr','May','Jun',
                'Jul','Ago','Sep','Oct','Nov','Dic']
monthly['Label'] = month_labels

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(monthly['Label'], monthly['Revenue']/1000,
              color=[NAVY if m < 10 else TEAL for m in monthly['Month']], width=0.6)
ax.set_title('Ventas Mensuales 2023 (en miles £)', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Ingresos (£ miles)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:.0f}k'))
for bar, val in zip(bars, monthly['Revenue']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'£{val/1000:.0f}k', ha='center', va='bottom', fontsize=8.5, color=NAVY)
ax.set_ylim(0, monthly['Revenue'].max()/1000 * 1.18)
plt.tight_layout()
plt.show()

best = monthly.loc[monthly['Revenue'].idxmax()]
worst = monthly.loc[monthly['Revenue'].idxmin()]
print(f'✦ Mejor mes:    {best["Label"]} — £{best["Revenue"]:,.0f}')
print(f'✦ Mes más bajo: {worst["Label"]} — £{worst["Revenue"]:,.0f}')
print(f'✦ Diferencia:   {best["Revenue"]/worst["Revenue"]:.1f}x entre pico y mínimo')

**Hallazgo:** Las ventas muestran una fuerte estacionalidad en Q4. Noviembre es el mes pico (+132% sobre el mínimo), impulsado por campañas de temporada (Black Friday, Navidad). Los meses de verano (Jul–Ago) son los más débiles.

## 4. Top 10 Productos por Ingresos

In [ ]:
top_prod = (df.groupby('Description')['TotalPrice']
              .sum()
              .sort_values(ascending=False)
              .head(10)
              .reset_index())
top_prod['Description'] = top_prod['Description'].str.title().str[:35]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_prod['Description'][::-1], top_prod['TotalPrice'][::-1]/1000, color=TEAL)
ax.set_title('Top 10 Productos por Ingresos', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Ingresos (£ miles)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:.0f}k'))
for bar, val in zip(bars, top_prod['TotalPrice'][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'£{val/1000:.1f}k', va='center', fontsize=8.5, color=NAVY)
ax.set_xlim(0, top_prod['TotalPrice'].max()/1000 * 1.18)
plt.tight_layout()
plt.show()

top3_rev = top_prod.head(3)['TotalPrice'].sum()
total_rev = df['TotalPrice'].sum()
print(f'✦ Top 3 productos representan el {top3_rev/total_rev*100:.1f}% del ingreso total')

## 5. Ventas por País (Mercados Internacionales)

In [ ]:
uk_rev = df[df['Country'] == 'United Kingdom']['TotalPrice'].sum()
intl_rev = df[df['Country'] != 'United Kingdom']['TotalPrice'].sum()
print(f'UK:            £{uk_rev:,.0f} ({uk_rev/total_rev*100:.1f}%)')
print(f'Internacional: £{intl_rev:,.0f} ({intl_rev/total_rev*100:.1f}%)')

top_countries = (df[df['Country'] != 'United Kingdom']
                   .groupby('Country')['TotalPrice']
                   .sum()
                   .sort_values(ascending=False)
                   .head(8)
                   .reset_index())

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(top_countries['Country'], top_countries['TotalPrice']/1000,
              color=AMBER, width=0.6)
ax.set_title('Top Países Internacionales (excluyendo UK)', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Ingresos (£ miles)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:.0f}k'))
plt.xticks(rotation=30, ha='right')
for bar, val in zip(bars, top_countries['TotalPrice']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'£{val/1000:.1f}k', ha='center', va='bottom', fontsize=8.5, color=NAVY)
plt.tight_layout()
plt.show()

## 6. Segmentación RFM de Clientes

In [ ]:
snapshot = df['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

def score_col(series, ascending=True):
    pct = series.rank(pct=True)
    if not ascending:
        pct = 1 - pct
    return pd.cut(pct, bins=5, labels=[1,2,3,4,5]).astype(float).fillna(1).astype(int)

rfm['R_score'] = score_col(rfm['Recency'], ascending=False)
rfm['F_score'] = score_col(rfm['Frequency'], ascending=True)
rfm['M_score'] = score_col(rfm['Monetary'], ascending=True)
rfm['RFM_Score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

def segment(score):
    if score >= 13:   return 'Champions'
    elif score >= 10: return 'Leales'
    elif score >= 7:  return 'Potenciales'
    elif score >= 4:  return 'En riesgo'
    else:             return 'Inactivos'

rfm['Segment'] = rfm['RFM_Score'].apply(segment)

seg_order  = ['Champions', 'Leales', 'Potenciales', 'En riesgo', 'Inactivos']
seg_colors = [TEAL, NAVY, AMBER, '#E07B54', GRAY]
seg_counts  = rfm['Segment'].value_counts().reindex(seg_order).fillna(0)
seg_revenue = rfm.groupby('Segment')['Monetary'].sum().reindex(seg_order).fillna(0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
wedges, texts, autotexts = ax1.pie(
    seg_counts, labels=seg_order, colors=seg_colors,
    autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
for t in autotexts: t.set_fontsize(9)
ax1.set_title('Distribución de Segmentos RFM', fontsize=13, fontweight='bold')

bars = ax2.bar(seg_order, seg_revenue/1000, color=seg_colors, width=0.6)
ax2.set_title('Ingresos por Segmento (£ miles)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Ingresos (£ miles)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:.0f}k'))
plt.setp(ax2.get_xticklabels(), rotation=20, ha='right')
for bar, val in zip(bars, seg_revenue):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'£{val/1000:.0f}k', ha='center', va='bottom', fontsize=8.5, color=NAVY)
plt.tight_layout()
plt.show()

champions_rev = seg_revenue['Champions']
total_rev_rfm = seg_revenue.sum()
print(f'✦ Champions ({seg_counts["Champions"]:.0f} clientes): £{champions_rev:,.0f} = {champions_rev/total_rev_rfm*100:.1f}% del ingreso')
print(f'✦ En riesgo ({seg_counts["En riesgo"]:.0f} clientes): £{seg_revenue["En riesgo"]:,.0f} — candidatos a campaña de retención')

## 7. Conclusiones y Recomendaciones

### Hallazgos principales

| # | Hallazgo | Impacto |
|---|---|---|
| 1 | Noviembre es el mes pico de ventas (+132% vs mínimo) | Priorizar inventario y personal en Q4 |
| 2 | El top 3 de productos concentra ~15% del ingreso | Proteger disponibilidad de estos SKUs |
| 3 | Alemania y Francia son los principales mercados internacionales | Potencial de expansión en Europa |
| 4 | Los Champions (16%) generan el 32.7% del ingreso | Programa de fidelidad para este segmento |
| 5 | 1,098 clientes están "En riesgo" con £214k en valor histórico | Campaña de reactivación inmediata |

### Recomendaciones

1. **Gestión de inventario:** Anticipar demanda Q4 desde agosto; los meses pico representan 2.3x el volumen de julio.
2. **Programa de fidelidad:** Crear beneficios exclusivos para Champions para mantener su frecuencia de compra.
3. **Reactivación:** Campaña de email marketing a los 1,098 clientes "En riesgo" con descuento personalizado.
4. **Expansión internacional:** Alemania y Francia muestran demanda sólida; evaluar traducción de catálogo y logística local.
5. **Estacionalidad de portafolio:** Diversificar hacia productos de primavera/verano para reducir la brecha Jul–Ago.